# Cleaned analysis notebook

This copy preserves the original analysis/calculation cells. Personal machine paths have been replaced with project-relative paths.
Place input workbooks and data in `data/`; generated files are written to `outputs/`. Create these folders before running.
Saved cell outputs were cleared so results are regenerated when the notebook is run.


In [ ]:

# ============================================================
# FOV ANALYSIS: 95% CONFIDENCE INTERVALS
# Complete vs Partial Field of View
# ============================================================

import numpy as np
import pandas as pd
from scipy.stats import fisher_exact, contingency
from statsmodels.stats.proportion import proportion_confint

# ------------------------------------------------------------
# 1. Enter observed counts
# ------------------------------------------------------------

complete_total = 208
complete_failures = 50

partial_total = 2
partial_failures = 1

complete_nonfailures = complete_total - complete_failures
partial_nonfailures = partial_total - partial_failures

# Contingency table: rows = FOV groups
# Columns = failures, non-failures
table = np.array([
    [partial_failures, partial_nonfailures],
    [complete_failures, complete_nonfailures]
])

print("Contingency table (Partial, Complete):")
print(pd.DataFrame(
    table,
    index=["Partial FOV", "Complete FOV"],
    columns=["Failure", "Non-failure"]
))

# ------------------------------------------------------------
# 2. Failure rates and 95% Wilson confidence intervals
# ------------------------------------------------------------

def calculate_rate_ci(failures, total):
    rate = failures / total
    lower, upper = proportion_confint(
        count=failures,
        nobs=total,
        alpha=0.05,
        method="wilson"
    )
    return rate * 100, lower * 100, upper * 100

for group, failures, total in [
    ("Complete FOV", complete_failures, complete_total),
    ("Partial FOV", partial_failures, partial_total)
]:
    rate, lower, upper = calculate_rate_ci(failures, total)
    print(
        f"\n{group}: {failures}/{total} "
        f"({rate:.2f}%; 95% CI {lower:.2f}%–{upper:.2f}%)"
    )

# ------------------------------------------------------------
# 3. Fisher's exact test
# ------------------------------------------------------------

# Rows: Partial FOV, Complete FOV
# Columns: Failure, Non-failure
fisher_or, fisher_p = fisher_exact(table, alternative="two-sided")

print(f"\nFisher's exact test p-value: {fisher_p:.4f}")
print(f"Cross-product odds ratio: {fisher_or:.4f}")

# ------------------------------------------------------------
# 4. Odds ratio and 95% confidence interval
# ------------------------------------------------------------

# Sample odds ratio and its 95% CI
sample_result = contingency.odds_ratio(table, kind="sample")
sample_ci = sample_result.confidence_interval(confidence_level=0.95)

print("\nSample odds ratio and 95% CI:")
print(f"OR = {sample_result.statistic:.4f}")
print(f"95% CI = {sample_ci.low:.4f} to {sample_ci.high:.4f}")

# Exact conditional odds ratio and 95% CI
exact_result = contingency.odds_ratio(table, kind="conditional")
exact_ci = exact_result.confidence_interval(confidence_level=0.95)

print("\nExact conditional odds ratio and 95% CI:")
print(f"OR = {exact_result.statistic:.4f}")
print(f"95% CI = {exact_ci.low:.4f} to {exact_ci.high:.4f}")
